# Bài 3: N-Step Lookahead — Thuật toán Minimax

**Dựa theo:** Kaggle Learn — *Intro to Game AI and Reinforcement Learning*, bài "N-Step Lookahead"
(gốc: https://www.kaggle.com/code/alexisbcook/n-step-lookahead)

**Nối tiếp:** `01_Play_the_Game_ConnectX_VN.ipynb`, `02_One_Step_Lookahead_ConnectX_VN.ipynb`

---

## Mục tiêu bài học

Ở bài 2, agent one-step lookahead chỉ nhìn trước **đúng 1 nước của chính mình**, hoàn toàn bỏ qua việc đối
thủ sẽ phản ứng ra sao. Bài này khắc phục hạn chế đó bằng thuật toán kinh điển trong Game AI: **Minimax**.

Sau bài này, bạn sẽ:

1. Hiểu ý tưởng **Minimax**: nhìn trước N nước, xen kẽ lượt "mình" (maximize điểm) và lượt "đối thủ"
   (minimize điểm — vì đối thủ cũng chơi tối ưu cho họ).
2. Hiểu khái niệm **cây trò chơi (game tree)** và cách Minimax duyệt cây này bằng đệ quy.
3. Viết được agent **N-step lookahead** hoàn chỉnh, có thể điều chỉnh độ sâu tìm kiếm.
4. Hiểu vấn đề **bùng nổ tổ hợp (combinatorial explosion)** khi N tăng, và vì sao cần giới hạn độ sâu
   (không thể tìm hết đến cuối ván như Tic-Tac-Toe).
5. (Mở rộng) Làm quen khái niệm **Alpha-Beta Pruning** — kỹ thuật cắt tỉa giúp Minimax chạy nhanh hơn.

> 💡 **Vì sao quan trọng cho đồ án?** Minimax là ví dụ kinh điển của **planning-based method** — agent
> "lập kế hoạch" bằng cách mô phỏng trước tương lai, khác hẳn với **model-free RL** (PPO/MAPPO) học chính
> sách trực tiếp từ trải nghiệm mà không cần mô phỏng cây quyết định. Hiểu rõ ưu/nhược của Minimax giúp bạn
> hiểu vì sao PPO/MAPPO là lựa chọn phù hợp hơn cho bài toán giao thông — nơi không gian trạng thái quá lớn
> để "nhìn trước" bằng cây quyết định như ConnectX.

## Phần 1 — Lý thuyết

### 1.1. Hạn chế của One-Step Lookahead

Xét tình huống: nếu agent đánh vào cột A, bàn cờ trông rất tốt (heuristic cao) — NHƯNG nước đó lại vô tình
"mở cửa" cho đối thủ thắng ngay ở nước tiếp theo. Agent one-step lookahead **không thấy được** điều này vì
nó chỉ nhìn 1 nước của chính mình, không mô phỏng tiếp nước đối thủ sẽ đánh trả.

### 1.2. Cây trò chơi (Game Tree)

Ta có thể biểu diễn tất cả các khả năng xảy ra dưới dạng 1 cây:

```
                    Bàn cờ hiện tại (lượt của TA)
                   /        |         \
              cột 0       cột 1      cột 2      ← các nước TA có thể đánh
              /  |  \      / | \      / | \
            c0 c1 c2  ...              ...      ← đối thủ đánh trả (lượt ĐỐI THỦ)
           /|\  ...
          ...                                    ← lại đến lượt TA (độ sâu tiếp theo)
```

- Các nút ở **độ sâu lẻ** (1, 3, 5...) là lượt của **đối thủ**.
- Các nút ở **độ sâu chẵn** (2, 4, 6...) là lượt lại của **ta** (nếu tính từ gốc là độ sâu 0).
- Nút lá (leaf node) — nơi ta dừng tìm kiếm — được chấm điểm bằng hàm heuristic giống bài 2.

### 1.3. Ý tưởng Minimax

Tên gọi "Minimax" đến từ chính chiến lược duyệt cây:

- Ở các nút thuộc lượt **của ta**: ta chọn nước đi cho điểm **cao nhất** trong số các nút con → **MAX**.
- Ở các nút thuộc lượt **của đối thủ**: ta giả định đối thủ luôn chọn nước có lợi nhất cho họ, tức là điểm
  **thấp nhất** (xét theo góc nhìn của ta) trong số các nút con → **MIN**.

Điểm số được "lan truyền ngược" (backpropagate) từ nút lá về gốc theo quy tắc MAX/MIN xen kẽ này. Cuối cùng,
ta chọn nước đi ở gốc dẫn tới nhánh có điểm MAX cao nhất.

> 🔑 **Trực giác:** Minimax trả lời câu hỏi *"Nếu tôi đánh nước này, và đối thủ chơi TỐI ƯU để chống lại tôi,
> thì kết quả tệ nhất tôi có thể nhận là gì?"* — rồi chọn nước đi mà "kết quả tệ nhất" đó vẫn tốt nhất có thể.
> Đây chính là nguyên lý phòng thủ chắc chắn (worst-case optimal), khác hẳn cách nghĩ "lạc quan" của
> one-step lookahead.

### 1.4. Độ sâu tìm kiếm (N) và bùng nổ tổ hợp

Số lượng nút trong cây trò chơi tăng theo cấp số nhân với độ sâu N: nếu trung bình có `b` nước đi khả dĩ mỗi
lượt (branching factor), cây có khoảng `b^N` nút. Với ConnectX (`b ≈ 7`), duyệt đến hết ván (~42 nước) là
bất khả thi về mặt tính toán. Vì vậy trong thực hành, ta:

- Chỉ tìm đến một **độ sâu N cố định** (ví dụ N=3), rồi dùng **heuristic** (như bài 2) để ước lượng điểm ở
  các nút chưa phải kết thúc ván.
- N càng lớn → agent "nhìn xa" hơn, chơi hay hơn, nhưng **chạy chậm hơn rất nhiều** (do `timeout` giới hạn
  ở mỗi lượt, không thể để N quá lớn).

### 1.5. (Mở rộng) Alpha-Beta Pruning

Alpha-Beta Pruning là kỹ thuật **cắt tỉa** giúp Minimax bỏ qua các nhánh chắc chắn không ảnh hưởng tới kết
quả cuối, giúp tìm kiếm nhanh hơn đáng kể mà **không làm thay đổi kết quả**. Ý tưởng: nếu trong quá trình
duyệt, ta phát hiện một nhánh đã đủ tệ để chắc chắn không được chọn (dù chưa duyệt hết), ta dừng duyệt nhánh
đó sớm. Bài này sẽ cài đặt Minimax cơ bản trước; phần mở rộng ở cuối bài sẽ giới thiệu cách thêm Alpha-Beta.

## Phần 2 — Thực hành

### 2.1. Cài đặt & tái sử dụng các hàm từ bài 2

Ta tái sử dụng `drop_piece`, `check_window`, `count_windows`, `get_heuristic` đã xây ở bài 2 (One-Step
Lookahead) — Minimax vẫn cần hàm heuristic để chấm điểm các nút lá.

In [ ]:
!pip install kaggle_environments -q


In [ ]:
from kaggle_environments import make, evaluate
import numpy as np
import random

env = make("connectx", debug=True)
config = env.configuration
print("Cấu hình môi trường:", config)


In [ ]:
# ==== Tái sử dụng từ bài 2 ====

def drop_piece(grid, col, mark, config):
    next_grid = grid.copy()
    for row in range(config.rows - 1, -1, -1):
        if next_grid[row][col] == 0:
            next_grid[row][col] = mark
            break
    return next_grid


def check_window(window, num_discs, piece, config):
    return (window.count(piece) == num_discs
            and window.count(0) == config.inarow - num_discs)


def count_windows(grid, num_discs, piece, config):
    num_windows = 0
    rows, columns, inarow = config.rows, config.columns, config.inarow

    for row in range(rows):
        for col in range(columns - inarow + 1):
            window = list(grid[row, col:col + inarow])
            if check_window(window, num_discs, piece, config):
                num_windows += 1

    for row in range(rows - inarow + 1):
        for col in range(columns):
            window = list(grid[row:row + inarow, col])
            if check_window(window, num_discs, piece, config):
                num_windows += 1

    for row in range(rows - inarow + 1):
        for col in range(columns - inarow + 1):
            window = [grid[row + i][col + i] for i in range(inarow)]
            if check_window(window, num_discs, piece, config):
                num_windows += 1

    for row in range(inarow - 1, rows):
        for col in range(columns - inarow + 1):
            window = [grid[row - i][col + i] for i in range(inarow)]
            if check_window(window, num_discs, piece, config):
                num_windows += 1

    return num_windows


def get_heuristic(grid, mark, config):
    opp_mark = 3 - mark
    num_fours       = count_windows(grid, 4, mark, config)
    num_threes      = count_windows(grid, 3, mark, config)
    num_twos        = count_windows(grid, 2, mark, config)
    num_threes_opp  = count_windows(grid, 3, opp_mark, config)
    num_fours_opp   = count_windows(grid, 4, opp_mark, config)

    score = (1e6  * num_fours
             + 1e2 * num_threes
             + 1   * num_twos
             - 1e2 * num_threes_opp
             - 1e6 * num_fours_opp)
    return score

print("Đã tái sử dụng xong các hàm từ bài 2.")


### 2.2. Hàm kiểm tra bàn cờ đã kết thúc chưa (`is_terminal_node`)

Trước khi cài Minimax, ta cần 1 hàm kiểm tra: bàn cờ hiện tại đã **kết thúc** chưa (có người thắng, hoặc
hết chỗ đánh = hoà)? Nếu là nút kết thúc (terminal node), ta không cần đệ quy sâu thêm nữa.

In [ ]:
def is_terminal_window(window, config):
    """Kiểm tra 1 cửa sổ có phải là 4-quân-liên-tiếp của người chơi 1 hoặc 2 không."""
    return window.count(1) == config.inarow or window.count(2) == config.inarow


def is_terminal_node(grid, config):
    """
    Trả về True nếu bàn cờ đã kết thúc: hết cột trống (hoà) HOẶC đã có người thắng.
    """
    # Hoà: không còn cột nào trống ở hàng trên cùng
    if list(grid[0, :]).count(0) == 0:
        return True

    rows, columns, inarow = config.rows, config.columns, config.inarow

    # Kiểm tra thắng theo 4 hướng, tương tự count_windows nhưng chỉ cần tìm 1 kết quả True
    for row in range(rows):
        for col in range(columns - inarow + 1):
            if is_terminal_window(list(grid[row, col:col + inarow]), config):
                return True

    for row in range(rows - inarow + 1):
        for col in range(columns):
            if is_terminal_window(list(grid[row:row + inarow, col]), config):
                return True

    for row in range(rows - inarow + 1):
        for col in range(columns - inarow + 1):
            window = [grid[row + i][col + i] for i in range(inarow)]
            if is_terminal_window(window, config):
                return True

    for row in range(inarow - 1, rows):
        for col in range(columns - inarow + 1):
            window = [grid[row - i][col + i] for i in range(inarow)]
            if is_terminal_window(window, config):
                return True

    return False


### 2.3. Thuật toán Minimax (đệ quy)

Đây là "trái tim" của bài học. Hàm `minimax` nhận vào:

- `node`: bàn cờ hiện tại (dạng ma trận 2D)
- `depth`: số nước còn được phép nhìn tiếp (giảm dần mỗi lần đệ quy, dừng khi = 0)
- `maximizingPlayer`: `True` nếu đang xét lượt của TA (muốn MAX điểm), `False` nếu là lượt đối thủ (muốn MIN điểm)
- `mark`, `config`: quân của ta và cấu hình trò chơi

Hàm trả về **điểm heuristic** của bàn cờ, giả định cả 2 bên đều chơi tối ưu từ đây trở đi (trong giới hạn
`depth` nước nhìn trước).

In [ ]:
def minimax(node, depth, maximizingPlayer, mark, config):
    """
    Trả về điểm heuristic của `node`, giả định cả 2 bên chơi tối ưu trong `depth` nước tiếp theo.
    """
    is_terminal = is_terminal_node(node, config)
    valid_moves = [c for c in range(config.columns) if node[0][c] == 0]

    # Điều kiện dừng đệ quy: hết độ sâu HOẶC bàn cờ đã kết thúc (thắng/thua/hoà)
    if depth == 0 or is_terminal:
        return get_heuristic(node, mark, config)

    if maximizingPlayer:
        # Lượt của TA: chọn nước cho điểm CAO NHẤT
        value = -np.Inf
        for col in valid_moves:
            child = drop_piece(node, col, mark, config)
            value = max(value, minimax(child, depth - 1, False, mark, config))
        return value
    else:
        # Lượt của ĐỐI THỦ: giả định họ chọn nước cho điểm THẤP NHẤT (xét theo góc nhìn của TA)
        opp_mark = 3 - mark
        value = np.Inf
        for col in valid_moves:
            child = drop_piece(node, col, opp_mark, config)
            value = min(value, minimax(child, depth - 1, True, mark, config))
        return value

print("Đã định nghĩa xong hàm minimax().")


### 2.4. Hàm chấm điểm 1 nước đi bằng Minimax và agent hoàn chỉnh

Tương tự `score_move` ở bài 2, nhưng lần này thay vì chấm điểm heuristic ngay sau 1 nước, ta gọi tiếp
`minimax` để nhìn sâu thêm `N_STEPS - 1` nước nữa (vì nước đầu tiên — nước của ta — đã được `drop_piece`
thực hiện trước khi gọi minimax).

In [ ]:
N_STEPS = 3   # Độ sâu tìm kiếm — có thể chỉnh 1, 3, 5... (số lẻ để nước cuối cùng luôn là lượt TA)


def score_move_minimax(grid, col, mark, config, nsteps):
    next_grid = drop_piece(grid, col, mark, config)
    # Sau khi TA đánh xong, đến lượt ĐỐI THỦ -> maximizingPlayer=False
    score = minimax(next_grid, nsteps - 1, False, mark, config)
    return score


def agent_n_step_lookahead(obs, config):
    grid = np.asarray(obs.board).reshape(config.rows, config.columns)
    valid_moves = [c for c in range(config.columns) if grid[0][c] == 0]

    scores = {col: score_move_minimax(grid, col, obs.mark, config, N_STEPS) for col in valid_moves}

    max_score = max(scores.values())
    best_moves = [col for col, s in scores.items() if s == max_score]
    return random.choice(best_moves)

print(f"Đã định nghĩa agent_n_step_lookahead với N_STEPS = {N_STEPS}")


### 2.5. So sánh với agent one-step lookahead ở bài 2

⚠️ **Lưu ý về tốc độ:** Minimax với `N_STEPS=3` chậm hơn one-step lookahead khá nhiều (do phải duyệt cây).
Nên bắt đầu với số ván (`n_rounds`) nhỏ để kiểm tra trước, tránh chờ quá lâu.

In [ ]:
def agent_random(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    return random.choice(valid_moves)


def agent_one_step_lookahead(obs, config):
    grid = np.asarray(obs.board).reshape(config.rows, config.columns)
    valid_moves = [col for col in range(config.columns) if grid[0][col] == 0]
    scores = {col: get_heuristic(drop_piece(grid, col, obs.mark, config), obs.mark, config) for col in valid_moves}
    max_score = max(scores.values())
    best_moves = [col for col, s in scores.items() if s == max_score]
    return random.choice(best_moves)


def get_win_percentages(agent1, agent2, n_rounds=20):
    cfg = {'rows': 6, 'columns': 7, 'inarow': 4}
    outcomes = evaluate("connectx", [agent1, agent2], cfg, [], n_rounds // 2)
    outcomes += [[b, a] for [a, b] in evaluate("connectx", [agent2, agent1], cfg, [], n_rounds - n_rounds // 2)]

    win_1 = np.round(outcomes.count([1, -1]) / len(outcomes) * 100, 1)
    win_2 = np.round(outcomes.count([-1, 1]) / len(outcomes) * 100, 1)
    draw = np.round(outcomes.count([0, 0]) / len(outcomes) * 100, 1)
    print(f"Agent 1 thắng: {win_1}%  |  Agent 2 thắng: {win_2}%  |  Hoà: {draw}%")


# Chạy với số ván nhỏ trước vì Minimax khá chậm
print("So sánh: n_step_lookahead (N=3) vs random")
get_win_percentages(agent_n_step_lookahead, agent_random, n_rounds=10)

print("\nSo sánh: n_step_lookahead (N=3) vs one_step_lookahead")
get_win_percentages(agent_n_step_lookahead, agent_one_step_lookahead, n_rounds=10)


> 🎯 **Kỳ vọng:** `agent_n_step_lookahead` nên thắng nhiều hơn `agent_one_step_lookahead`, vì nó biết
> tránh những nước đi "trông tốt trước mắt nhưng mở đường cho đối thủ thắng" — điều mà one-step lookahead
> không thấy được.

In [ ]:
# Xem lại 1 ván đấu cụ thể (chạy có thể mất vài giây do Minimax cần tính toán)
env.run([agent_n_step_lookahead, agent_random])
env.render(mode="ipython", width=500, height=450)
# Nếu không hiển thị được HTML, dùng: print(env.render(mode="ansi"))


## Phần 3 — Bài tập thực hành

### Bài tập 1: Thử nghiệm với độ sâu khác nhau

Đổi `N_STEPS` thành `1` (nên giống hệt one-step lookahead), rồi thử `5`. Quan sát:
- Tỷ lệ thắng có tăng theo N không?
- Thời gian chạy `env.run()` thay đổi ra sao? (Gợi ý: dùng `%%time` ở đầu cell để đo thời gian.)

### Bài tập 2 (nâng cao): Cài đặt Alpha-Beta Pruning

Hãy thử tự bổ sung 2 tham số `alpha` và `beta` vào hàm `minimax` để cắt tỉa các nhánh không cần thiết. Ý
tưởng cơ bản:

```python
def minimax_ab(node, depth, alpha, beta, maximizingPlayer, mark, config):
    ...
    if maximizingPlayer:
        value = -np.Inf
        for col in valid_moves:
            child = drop_piece(node, col, mark, config)
            value = max(value, minimax_ab(child, depth - 1, alpha, beta, False, mark, config))
            alpha = max(alpha, value)
            if alpha >= beta:
                break   # cắt tỉa: nhánh này chắc chắn không được chọn, dừng sớm
        return value
    else:
        value = np.Inf
        opp_mark = 3 - mark
        for col in valid_moves:
            child = drop_piece(node, col, opp_mark, config)
            value = min(value, minimax_ab(child, depth - 1, alpha, beta, True, mark, config))
            beta = min(beta, value)
            if alpha >= beta:
                break
        return value
```

Gọi hàm ban đầu với `alpha=-np.Inf, beta=np.Inf`. Kết quả trả về phải **giống hệt** Minimax gốc, nhưng chạy
**nhanh hơn** — hãy đo và so sánh thời gian chạy giữa 2 phiên bản với cùng `N_STEPS`.

In [ ]:
# TODO (Bài tập 2): cài đặt minimax_ab tại đây, rồi viết agent_n_step_ab dùng nó,
# so sánh thời gian chạy với agent_n_step_lookahead (dùng %%time hoặc module time)

import time

start = time.time()
get_win_percentages(agent_n_step_lookahead, agent_random, n_rounds=6)
print(f"Thời gian chạy Minimax gốc: {time.time() - start:.2f} giây")

# Sau khi cài xong minimax_ab, so sánh tương tự để thấy alpha-beta pruning nhanh hơn bao nhiêu


## Phần 4 — Liên hệ với đồ án PPO/MAPPO điều khiển đèn giao thông

| Trong N-Step Lookahead (Minimax) | Trong đồ án điều khiển đèn giao thông |
|---|---|
| Cây trò chơi mô phỏng mọi khả năng đến độ sâu N | Không gian trạng thái giao thông là **liên tục và cực lớn** (mật độ xe, vị trí từng xe...) → không thể liệt kê cây quyết định như ConnectX |
| Minimax giả định đối thủ (2 người chơi) luân phiên | Trong bài toán 1 giao lộ: chỉ có 1 "agent" ra quyết định (không có đối thủ đối kháng) → không cần cấu trúc MIN/MAX |
| **MAPPO** (Multi-Agent PPO) cho nhiều giao lộ | Ở đây "nhiều agent" (nhiều đèn giao thông) lại **hợp tác** (cooperative), không đối kháng — khác bản chất với Minimax (2 người chơi đối kháng zero-sum) |
| Bùng nổ tổ hợp khi N tăng (`b^N` nút) | Chính là lý do vì sao **model-free RL** (PPO) được chọn thay vì planning/search: PPO học 1 **policy network** xấp xỉ trực tiếp hành động tốt, không cần duyệt cây tại mỗi bước quyết định thời gian thực |
| Alpha-Beta Pruning giúp tìm kiếm nhanh hơn mà kết quả không đổi | Ý tưởng "cắt giảm tính toán không cần thiết" cũng xuất hiện trong RL, ví dụ **early stopping**, **experience replay có ưu tiên (prioritized replay)**, hay giới hạn horizon trong GAE (Generalized Advantage Estimation) |

> ✅ **Điểm mấu chốt cần rút ra:** Minimax hoạt động tốt cho game 2 người, không gian trạng thái hữu hạn,
> có thể mô phỏng nhanh mỗi nước đi. Bài toán giao thông của bạn **không thoả** các điều kiện đó (nhiều agent
> hợp tác, không gian trạng thái liên tục, không thể "dừng thời gian" để duyệt cây mỗi khi cần ra quyết
> định) — đây chính là lý do PPO/MAPPO (model-free, học policy trực tiếp qua trải nghiệm) là lựa chọn thuật
> toán phù hợp hơn nhiều so với các phương pháp tìm kiếm cây như Minimax.

---

## Tóm tắt bài học

- ✅ Hiểu khái niệm **cây trò chơi (game tree)** và chiến lược **Minimax** (MAX cho ta, MIN cho đối thủ).
- ✅ Cài đặt được hàm `is_terminal_node` để phát hiện bàn cờ đã kết thúc.
- ✅ Cài đặt được thuật toán **minimax()** đệ quy hoàn chỉnh và agent N-step lookahead.
- ✅ Hiểu vấn đề **bùng nổ tổ hợp** và lý do phải giới hạn độ sâu tìm kiếm N.
- ✅ Biết ý tưởng cơ bản của **Alpha-Beta Pruning** để tăng tốc Minimax.
- ✅ Hiểu rõ vì sao các phương pháp planning/search (Minimax) không phù hợp với bài toán RL nhiều agent,
  không gian trạng thái liên tục như điều khiển đèn giao thông — từ đó củng cố lý do chọn PPO/MAPPO.

**Bài tiếp theo trong khoá học gốc:** *Deep Reinforcement Learning* — thay vì viết tay heuristic, ta để một
mạng neural network **tự học** hàm giá trị / chính sách qua quá trình huấn luyện bằng RL (ví dụ Proximal
Policy Optimization — PPO), đây chính là bước chuyển tiếp trực tiếp sang phần lõi thuật toán trong đồ án
của bạn.